In [3]:
from nilearn import plotting
%matplotlib inline
from os.path import join as opj
import json
from nipype.interfaces.base import Bunch
from nipype.interfaces.spm import Level1Design, EstimateModel, EstimateContrast, SPMCommand, Info, model
from nipype.interfaces.matlab import MatlabCommand
from nipype.interfaces.freesurfer import FSCommand
from nipype.algorithms.modelgen import SpecifySPMModel, SpecifyModel
from nipype.interfaces.utility import Function, IdentityInterface
from nipype.interfaces.io import SelectFiles, DataSink
from nipype import Workflow, Node
from bids.layout import BIDSLayout
from glob import glob
from scipy import io, stats
from itertools import chain
import pandas as pd
import numpy as np
#import pytest as pt # not even needed
import nibabel as nb
import nipype
import os.path as op
from nipype.interfaces import spm



In [ ]:
# for local
bids_folder = '/Volumes/mrenkeED/data/ds-stressrisk'

MatlabCommand.set_default_matlab_cmd('/Applications/MATLAB_R2021b.app/bin/matlab')#'/Users/mrenke/matlab') # /Users/mrenke/matlab
spm.SPMCommand.set_mlab_paths(matlab_cmd="/Applications/MATLAB_R2021b.app/bin/matlab") # error during spm.EstimateModel, solved: https://neurostars.org/t/nipype-spm-spmcommand-version-does-not-return-anything-on-mac/2871

print(spm.SPMCommand().version)

In [7]:
# for sciencecloud2
MatlabCommand.set_default_paths('/home/ubuntu/matlab/spm12') # 
bids_folder = '/mnt_01/ds-stressrisk' 
print(spm.SPMCommand().version)

12.7771


In [8]:
sub='01'
ses = 1
run = 1

with open(op.join(bids_folder, 'sub-01', 'ses-1','func', 'sub-01_ses-1_task-risk_run-1_bold.json'), # 'derivatives/fmriprep' , 
    "rt",
) as fp:
    task_info = json.load(fp)
TR = task_info["RepetitionTime"]
Nslices = len(task_info['SliceTiming']) # = 39
refSlice = 20 #Nslices / 2

In [97]:
def get_subject_info(subject):
    from glob import glob
    import numpy as np
    import pandas as pd
    from scipy import io, stats
    from nipype.interfaces.base import Bunch
    import os.path as op
    import os 

    bids_folder='/mnt_01/ds-stressrisk'     
    
    sub = subject
    subject_info = []

    functional_runs = []
    
    for ses in [1,2]:
        for run in range(1, 7):
            run_ses_i = (ses - 1)*6 + run
            
            # regressors (confounds + physio)
            confounds = pd.read_csv(op.join(bids_folder, 'derivatives/fmriprep',f'sub-{sub}', f'ses-{ses}', 'func', 
                            f'sub-{sub}_ses-{ses}_task-risk_run-{run}_desc-confounds_timeseries.tsv'), sep='\t')
            confound_names = ["trans_x","trans_y","trans_z","rot_x","rot_y","rot_z","a_comp_cor_00","a_comp_cor_02","a_comp_cor_03","a_comp_cor_04"]
            confounds = confounds.loc[:, confound_names]
            fn_physio = op.join(bids_folder, 'derivatives/physiotoolbox',f'sub-{sub}', f'ses-{ses}', 'func', 
                                f'sub-{sub}_ses-{ses}_task-task_run-{run}_desc-retroicor_output.mat') # task-taks (not risk) 
            physio = io.loadmat(fn_physio, simplify_cells=True)["physio"]["model"]
            physio = pd.DataFrame(
                data=physio["R"],
                columns=physio["R_column_names"])
            regressors = pd.concat([confounds, physio], axis=1)
            regressor_names = regressors.columns.values.tolist()

            # events + pmods
            df_events = pd.read_csv(op.join(bids_folder, f'sub-{sub}', f'ses-{ses}', 'func', f'sub-{sub}_ses-{ses}_task-risk_run-{run}_events.tsv'), sep='\t')
            df_events.set_index(['trial_nr', 'trial_type'], inplace=True) # only look at second option
            
            df_events_num1 = df_events.xs('stimulus 1',0,'trial_type')[['onset','prob1','n1']]
            df_risky_num1 = df_events_num1[df_events_num1['prob1'] == 0.55]
            df_safe_num1 = df_events_num1[df_events_num1['prob1'] == 1]   
            
            df_events_num2 = df_events.xs('stimulus 2',0,'trial_type')[['onset','prob2','n2']]
            df_risky_num2 = df_events_num2[df_events_num2['prob2'] == 0.55]
            df_safe_num2 = df_events_num2[df_events_num2['prob2'] == 1]

            pmod = [
                Bunch(name=['num1_risky'], param=[df_risky_num1['n1'].values.tolist()], poly=[1]),
                Bunch(name=['num1_safe'],param=[df_safe_num1['n1'].values.tolist()], poly=[1]),
                Bunch(name=['num2_risky'], param=[df_risky_num2['n2'].values.tolist()], poly=[1]),
                Bunch(name=['num2_safe'],param=[df_safe_num2['n2'].values.tolist()], poly=[1])]
            onsets = [
                df_risky_num1['onset'].values.tolist(), 
                df_safe_num1['onset'].values.tolist(),
                df_risky_num2['onset'].values.tolist(), 
                df_safe_num2['onset'].values.tolist()]

            durations = [(np.ones(len(df_risky_num1))* 0.6).tolist(), 
                         (np.ones(len(df_safe_num1))* 0.6).tolist(),
                         (np.ones(len(df_risky_num2))* 0.6).tolist(), 
                         (np.ones(len(df_safe_num2))* 0.6).tolist()]

            # put all together
            conditions = [f'risky_num1_ses{ses}', f'safe_num1_ses{ses}', f'risky_num2_ses{ses}', f'safe_num2_ses{ses}']
            subject_info.insert(
                run_ses_i - 1,
                Bunch(
                    conditions=conditions,
                    onsets=onsets,
                    durations=durations,
                    pmod=pmod,
                    tmod=None,
                    orth=['No']*len(conditions),
                    regressors=regressors.values.T.tolist(),
                    regressor_names=regressor_names,
                ),
            )

            nifti_file =  op.join(bids_folder,'derivatives/spm_nipype', f'sub-{sub}', f'ses-{ses}', f'ssub-{sub}_ses-{ses}_task-risk_run-{run}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii')
            
            #print(f'{nifti_file}')

            if op.isfile(nifti_file):
                functional_run = glob(nifti_file)[0]
                functional_runs.append(functional_run)
            else:
                print(f'{nifti_file} does not exist, probably smoothing did not work')

    return subject_info, functional_runs



In [100]:
# try function
sub = '10'
subject_info, functional_runs = get_subject_info(sub)
functional_runs

/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-2_space-MNI152NLin2009cAsym_desc-preproc_bold.nii does not exist, probably smoothing did not work
/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-2/ssub-10_ses-2_task-risk_run-3_space-MNI152NLin2009cAsym_desc-preproc_bold.nii does not exist, probably smoothing did not work


['/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-3_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-4_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-5_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-1/ssub-10_ses-1_task-risk_run-6_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-2/ssub-10_ses-2_task-risk_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-10/ses-2/ssub-10_ses-2_task-risk_run-2_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/mnt_01/ds-stressr

In [67]:
def get_contrasts(subject_info):
    from nipype.interfaces.spm import EstimateContrast
    import os

    pmod_names = ['risky_num2_ses1xnum2_risky^1','risky_num2_ses2xnum2_risky^1', # 'group by riksy/safe for easier indexiing
                'safe_num2_ses1xnum2_safe^1','safe_num2_ses2xnum2_safe^1']
    condition_names = ['risky_num2_ses1','risky_num2_ses2','safe_num2_ses1','safe_num2_ses2']
    
    # same for both sessions
    con01 = ('num2_risky_int_bothSes', 'T', [condition_names[0:2]], [1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con02 = ('num2_safe_int_bothSes', 'T', [condition_names[2:4]], [1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]  
    con03 = ('num2_risky_pmod_bothSes', 'T', [pmod_names[0:2]], [1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con04 = ('num2_safe_pmod_bothSes', 'T', [pmod_names[2:4]], [1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]
    
    # difference between sessions
    con05 = ('num2_risky_int_sesDif', 'T', [condition_names[0:2]], [-1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con06 = ('num2_safe_int_sesDif', 'T', [condition_names[2:4]], [-1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]  
    con07 = ('num2_risky_sesDif', 'T', [pmod_names[0:2]], [-1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con08 = ('num2_safe_sesDif', 'T', [pmod_names[2:4]], [-1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]
       
    con_list = [con01, con02, con03, con04, con05,con06, con07, con08]
    
    return con_list


### start the workflow with runnning single nodes

In [76]:

getsubjectinfo = Node(
    Function(
        input_names=["subject"],
        output_names=["subject_info", "functional_runs"],
        function=get_subject_info,
    ),
    name="getsubjectinfo",
)

getsubjectinfo.inputs.subject = '03'
getsubjectinfo = getsubjectinfo.run()

240311-14:57:40,396 nipype.workflow INFO:
	 [Node] Setting-up "getsubjectinfo" in "/tmp/tmp0pl0qoaq/getsubjectinfo".
240311-14:57:40,398 nipype.workflow INFO:
	 [Node] Executing "getsubjectinfo" <nipype.interfaces.utility.wrappers.Function>
240311-14:57:40,653 nipype.workflow INFO:
	 [Node] Finished "getsubjectinfo", elapsed time 0.253295s.
240311-14:57:40,655 nipype.workflow WARNING:
	 Storing result file without outputs
240311-14:57:40,656 nipype.workflow WARNING:
	 [Node] Error on "getsubjectinfo" (/tmp/tmp0pl0qoaq/getsubjectinfo)


NodeExecutionError: Exception raised while executing Node getsubjectinfo.

Traceback:
	Traceback (most recent call last):
	  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/interfaces/base/core.py", line 397, in run
	    runtime = self._run_interface(runtime)
	  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/interfaces/utility/wrappers.py", line 142, in _run_interface
	    out = function_handle(**args)
	  File "<string>", line 79, in get_subject_info
	IndexError: list index out of range


In [71]:
modelspec = Node(
    SpecifySPMModel(
        concatenate_runs=False,
        input_units="secs",
        output_units="secs", # 'scans' ??
        time_repetition=TR,
        high_pass_filter_cutoff=128,
    ),
    name="modelspec",
)

modelspec.inputs.subject_info = getsubjectinfo.outputs.subject_info
modelspec.inputs.functional_runs = getsubjectinfo.outputs.functional_runs

modelspec = modelspec.run()
print(modelspec.outputs.session_info)

240311-14:48:19,173 nipype.workflow INFO:
	 [Node] Setting-up "modelspec" in "/tmp/tmpjegg7syp/modelspec".
240311-14:48:19,177 nipype.workflow INFO:
	 [Node] Executing "modelspec" <nipype.algorithms.modelgen.SpecifySPMModel>
240311-14:48:19,178 nipype.workflow WARNING:
	 [Node] Error on "modelspec" (/tmp/tmpjegg7syp/modelspec)


ValueError: SpecifySPMModel requires a value for one of the inputs 'subject_info, event_files, bids_event_file'. For a list of required inputs, see SpecifySPMModel.help()

In [72]:
# Level1Design - Generates an SPM design matrix
level1design = Node(
    Level1Design(
        bases={"hrf": {"derivs": [0, 0]}},
        timing_units="secs",
        interscan_interval=TR,
        model_serial_correlations="AR(1)",
        microtime_resolution=Nslices,
        microtime_onset=refSlice,
        flags={"mthresh": 0.8, "globalnorm": 'None'},
        #mask_image="/mnt/d/multlearn-sns/SPM/mask_ICV.nii", ??
        volterra_expansion_order=1,
    ),
    name="level1design",
)

level1design.inputs.session_info = modelspec.outputs.session_info
level1design = level1design.run()
print(level1design.outputs.spm_mat_file)

240311-14:48:20,446 nipype.workflow INFO:
	 [Node] Setting-up "level1design" in "/tmp/tmp5qthfjue/level1design".
240311-14:48:20,450 nipype.workflow INFO:
	 [Node] Executing "level1design" <nipype.interfaces.spm.model.Level1Design>
240311-14:48:20,451 nipype.workflow WARNING:
	 [Node] Error on "level1design" (/tmp/tmp5qthfjue/level1design)


ValueError: Level1Design requires a value for input 'session_info'. For a list of required inputs, see Level1Design.help()

In [16]:
# EstimateModel - estimate the parameters of the model
level1estimate = Node( EstimateModel(estimation_method={"Classical": 1},write_residuals=False), name="level1estimate")

level1estimate.inputs.spm_mat_file = level1design.outputs.spm_mat_file
level1estimate = level1estimate.run()

240311-13:17:07,293 nipype.workflow INFO:
	 [Node] Setting-up "level1estimate" in "/tmp/tmphwv62ul5/level1estimate".
240311-13:17:07,307 nipype.workflow INFO:
	 [Node] Executing "level1estimate" <nipype.interfaces.spm.model.EstimateModel>


stty: 'standard input': Inappropriate ioctl for device


240311-13:20:43,741 nipype.workflow INFO:
	 [Node] Finished "level1estimate", elapsed time 216.431175s.


In [17]:
def get_contrasts(subject_info):
    from nipype.interfaces.spm import EstimateContrast
    import os

    pmod_names = ['risky_num2_ses1xnum2_risky^1','risky_num2_ses2xnum2_risky^1', # 'group by riksy/safe for easier indexiing
                'safe_num2_ses1xnum2_safe^1','safe_num2_ses2xnum2_safe^1']
    condition_names = ['risky_num2_ses1','risky_num2_ses2','safe_num2_ses1','safe_num2_ses2']
    
    # same for both sessions
    con01 = ('num2_risky_int_bothSes', 'T', condition_names[0:2], [1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con02 = ('num2_safe_int_bothSes', 'T', condition_names[2:4], [1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]  
    con03 = ('num2_risky_pmod_bothSes', 'T', pmod_names[0:2], [1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con04 = ('num2_safe_pmod_bothSes', 'T', pmod_names[2:4], [1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]
    
    # difference between sessions
    con05 = ('num2_risky_int_sesDif', 'T', condition_names[0:2], [-1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con06 = ('num2_safe_int_sesDif', 'T', condition_names[2:4], [-1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]  
    con07 = ('num2_risky_sesDif', 'T', pmod_names[0:2], [-1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con08 = ('num2_safe_sesDif', 'T', pmod_names[2:4], [-1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]
       
    con_list = [con01, con02, con03, con04, con05,con06, con07, con08]
    
    return con_list

In [ ]:
# EstimateContrast - estimates contrasts
getcontrasts = Node(Function(
        input_names=["subject_info"],
        output_names=["contrasts"],
        function=get_contrasts),
        name="getcontrasts")

getcontrasts.inputs.subject_info = getsubjectinfo.outputs.subject_info # output from very first node
getcontrasts = getcontrasts.run()

level1conest = Node(EstimateContrast(), name="level1conest")
level1conest.inputs.contrasts =  getcontrasts.outputs.contrasts
level1conest.inputs.spm_mat_file = level1estimate.outputs.spm_mat_file
level1conest.inputs.beta_images = level1estimate.outputs.beta_images
level1conest.inputs.residual_image = level1estimate.outputs.residual_image

level1conest = level1conest.run()
print(level1conest.outputs)

## connect all Nodes to a workflow

In [50]:
getsubjectinfo = Node(
    Function(
        input_names=["subject"],
        output_names=["subject_info", "functional_runs"],
        function=get_subject_info,
    ),
    name="getsubjectinfo")

modelspec = Node(
    SpecifySPMModel(
        concatenate_runs=False,
        input_units="secs",
        output_units="secs", # 'scans' ??
        time_repetition=TR,
        high_pass_filter_cutoff=128,
    ),
    name="modelspec")

level1design = Node(
    Level1Design(
        bases={"hrf": {"derivs": [0, 0]}},
        timing_units="secs",
        interscan_interval=TR,
        model_serial_correlations="AR(1)",
        microtime_resolution=Nslices,
        microtime_onset=refSlice,
        flags={"mthresh": 0.8, "globalnorm": 'None'},
        #mask_image="/mnt/d/multlearn-sns/SPM/mask_ICV.nii", ??
        volterra_expansion_order=1,),
        name="level1design")

level1estimate = Node( 
    EstimateModel(estimation_method={"Classical": 1},write_residuals=False),
     name="level1estimate")

getcontrasts = Node(Function(
        input_names=["subject_info"],
        output_names=["contrasts"],
        function=get_contrasts),
        name="getcontrasts")

level1conest = Node(
    EstimateContrast(), name="level1conest")


In [62]:
# Create a Nipype workflow
first_level_wf = Workflow(name='first_level_wf')

getsubjectinfo.inputs.subject = '01'

# Connect the nodes
first_level_wf.connect([
    (getsubjectinfo, modelspec, [('subject_info', 'subject_info'), ('functional_runs', 'functional_runs')]),
    (modelspec, level1design, [('session_info', 'session_info')]),
    (getsubjectinfo, getcontrasts, [('subject_info', 'subject_info')]),
    (level1design, level1estimate, [('spm_mat_file', 'spm_mat_file')]),  # Connect level1design to level1estimate
    (getcontrasts, level1conest, [('contrasts', 'contrasts')]),  # Connect the contrasts to EstimateContrast
    (level1design, level1conest, [('spm_mat_file', 'spm_mat_file')])
])


first_level_wf.config['logging'] = {'workflow_level' : 'DEBUG',
                        'filemanip_level' : 'DEBUG',
                        'interface_level' : 'DEBUG',
                        'log_to_file' : 'True',
                        'log_directory' : '/output/log_folder'}

first_level_wf.run()

240311-14:26:25,641 nipype.workflow INFO:
	 Workflow first_level_wf settings: ['check', 'execution', 'logging', 'monitoring']
240311-14:26:25,649 nipype.workflow INFO:
	 Running serially.
240311-14:26:25,650 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.getsubjectinfo" in "/tmp/tmp04c662u5/first_level_wf/getsubjectinfo".
240311-14:26:25,652 nipype.workflow INFO:
	 [Node] Executing "getsubjectinfo" <nipype.interfaces.utility.wrappers.Function>
240311-14:26:26,394 nipype.workflow INFO:
	 [Node] Finished "getsubjectinfo", elapsed time 0.74071s.
240311-14:26:26,462 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.modelspec" in "/tmp/tmpeq89p9le/first_level_wf/modelspec".
240311-14:26:26,787 nipype.workflow INFO:
	 [Node] Executing "modelspec" <nipype.algorithms.modelgen.SpecifySPMModel>
240311-14:26:26,795 nipype.workflow INFO:
	 [Node] Finished "modelspec", elapsed time 0.006157s.
240311-14:26:27,167 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.getcontras

stty: 'standard input': Inappropriate ioctl for device


240311-14:27:14,465 nipype.workflow INFO:
	 [Node] Executing "level1estimate" <nipype.interfaces.spm.model.EstimateModel>


stty: 'standard input': Inappropriate ioctl for device


240311-14:30:51,894 nipype.workflow INFO:
	 [Node] Finished "level1estimate", elapsed time 217.426989s.
240311-14:30:51,913 nipype.workflow INFO:
	 [Node] Setting-up "first_level_wf.level1conest" in "/tmp/tmp2f77po9q/first_level_wf/level1conest".
240311-14:30:51,967 nipype.workflow INFO:
	 [Node] Executing "level1conest" <nipype.interfaces.spm.model.EstimateContrast>
240311-14:30:51,968 nipype.workflow WARNING:
	 [Node] Error on "first_level_wf.level1conest" (/tmp/tmp2f77po9q/first_level_wf/level1conest)
240311-14:30:51,969 nipype.workflow ERROR:
	 Node level1conest failed to run on host mr-02.
240311-14:30:51,970 nipype.workflow ERROR:
	 Saving crash info to /home/ubuntu/git/stress_risk/stress_risk/fmri_analysis/glm_nipype/crash-20240311-143051-ubuntu-level1conest-9fcc54f7-2a2a-4e7d-a460-ebc05e7b079d.pklz
Traceback (most recent call last):
  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/pipeline/plugins/linear.py", line 47, in run
    node.run(updat

ValueError: EstimateContrast requires a value for input 'beta_images'. For a list of required inputs, see EstimateContrast.help()

In [58]:
crash_file = glob('*-level1conest-*.pklz')
crash_file

['crash-20240311-141654-ubuntu-level1conest-1d8d9145-5d12-42ff-b769-69e6d803907d.pklz']

In [60]:
from nipype.utils.filemanip import loadpkl

crash_file = glob('*-level1conest-*.pklz')
#' /home/ubuntu/git/stress_risk/stress_risk/fmri_analysis/glm_nipype/crash-20240311-141654-ubuntu-level1conest-1d8d9145-5d12-42ff-b769-69e6d803907d.pklz'
res = loadpkl(crash_file[0])

In [61]:
res

{'node': first_level_wf.level1conest,
 'traceback': ['Traceback (most recent call last):\n',
  '  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/pipeline/plugins/linear.py", line 47, in run\n    node.run(updatehash=updatehash)\n',
  '  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/pipeline/engine/nodes.py", line 527, in run\n    result = self._run_interface(execute=True)\n',
  '  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/pipeline/engine/nodes.py", line 645, in _run_interface\n    return self._run_command(execute)\n',
  '  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/pipeline/engine/nodes.py", line 722, in _run_command\n    result = self._interface.run(cwd=outdir, ignore_exception=True)\n',
  '  File "/home/ubuntu/miniforge3/envs/nipype_spm/lib/python3.10/site-packages/nipype/interfaces/base/core.py", line 388, in run\n    self._check_mandat

In [41]:
sub = '01'
ses=1
run=1
nifti_file =  op.join(bids_folder,'derivatives/spm_nipype', f'sub-{sub}', f'ses-{ses}', f'ssub-{sub}_ses-{ses}_task-risk_run-{run}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii')
functional_run = glob(nifti_file)[0]
functional_run

'/mnt_01/ds-stressrisk/derivatives/spm_nipype/sub-01/ses-1/ssub-01_ses-1_task-risk_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii'